# improve

> Replay past sessions, and keep the edits to Ramabana's learned folder that make the agent work better.

In [ ]:
#| default_exp improve

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

Ramabana loads a learned folder into every session: `AGENTS.md`, a few short rules included in every briefing, and skills it opens when a task calls for them. A round tries to improve that folder from one finished session, in the manner of the propose, probe and compose step of Harness². It replays the session's request `k` times from the commit the session started at, under the current folder. An editor model reads those runs and edits a copy of the folder. The request is replayed once more under the copy. The copy is kept only when that run replied, took a cheaper route than every base run, and a judge model rated its reply at least as good as the best base reply. Each kept copy is committed in the learned folder's own git repository, one commit per round.

In [ ]:
#| export
import json, os, random, re, shutil, subprocess, sys
from collections import Counter
from contextlib import contextmanager, nullcontext
from datetime import datetime, timezone
from fastcore.docments import frontmatter
from fastcore.script import call_parse
from fastcore.xtras import Path

from dhrona.core import read_history, assess_turn, tool_args, denial_reason

RAMABANA = 'ramabana'   #: the command a round runs; another install replays with that one
LEARNED_MAX = 2000      #: characters of always-on `AGENTS.md` a candidate may carry
_HIDDEN = shutil.ignore_patterns('.*')   # `.git`, and the `.kosha` index Ramabana writes into every folder it opens

## Runs

`replay_clone` makes a throwaway clone of the repository at the commit a session started from. The clone shares no object files with the source and has no remote, so a replay can neither change nor push to the original. `run_turn` runs one Ramabana turn with no interface, in a fresh config folder, so no history, extension, skill or project note from the person's own config takes part. It reads the turn back from that folder's history, which records the request, every tool call and the reply, and leaves Ramabana's output in `run.log`. Writes are refused unless `approve` allows them. A round's replays pass `read_outside`, so they can read installed packages and other folders the way an interactive session does, while their writes stay inside their clone.

In [ ]:
#| export
def replay_clone(start, dest):
    "A throwaway clone of `start['repo']` checked out at `start['head']`, in `dest`."
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    for cmd in (['git', 'clone', '-q', '--no-hardlinks', start['repo'], str(dest)],   # a replay's edits never reach the source's objects
                ['git', '-C', str(dest), 'checkout', '-q', '--detach', start['head']],
                ['git', '-C', str(dest), 'remote', 'remove', 'origin']):   # a push from a replay has nowhere to go
        if (p := subprocess.run(cmd, capture_output=True, text=True)).returncode:
            raise RuntimeError(f"could not clone {start['repo']} at {start['head']}: {p.stderr.strip()}")
    return dest

def run_turn(
    prompt,          # the request
    roots,           # folders the turn may work in
    cfg,             # a fresh config folder for this one turn
    model=None,      # Ramabana's routing default when None
    learned=None,    # a learned folder to copy in as `<cfg>/learned`
    approve='ask',   # `ask` refuses every write in a headless turn; `edits` allows file edits
    exe=RAMABANA,
    timeout=1800,    # seconds before the run is abandoned
    read_outside=False, # reads may name any path, as in an interactive session; writes stay in `roots`
):
    "Run one Ramabana turn with no interface and return its history row: `prompt`, `activity`, `reply`, `usage`."
    cfg = Path(cfg)
    cfg.mkdir(parents=True, exist_ok=True)
    if learned is not None: shutil.copytree(learned, cfg/'learned', ignore=_HIDDEN)
    cmd = [exe, '--root', ','.join(map(str, roots)), '--cfg', str(cfg), '--approve', approve,
           '--no-web', '--tmux', 'off', '--pane', 'off', *(['--read-outside'] if read_outside else []),
           *(['--model', model] if model else []), prompt]
    try: p = subprocess.run(cmd, stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired: raise RuntimeError(f'the run in {cfg} was still going after {timeout}s')
    (cfg/'run.log').write_text(f'exit {p.returncode}\n\n{p.stderr}')
    hist = cfg/'agent-history.jsonl'
    if not hist.exists() or not (turns := read_history(hist)): raise RuntimeError(f'no turn was recorded; see {cfg/"run.log"}')
    return turns[-1]

## Comparing runs

A route's cost is a tuple of its route findings, its failed calls and its calls, and a smaller tuple is a cheaper route. A replay has nobody to answer approvals, so a refused approval does not count as a failure and asking again does not count as a `denial_retry`; in a real session a person would have answered. `delivered` says whether a run produced an answer at all: at least one model turn, a reply, and no error. `better` passes a candidate only when it delivered and costs less than every base run. Tying the best base run is not enough.

A cheaper route alone proves little, because an agent that gives up early is cheap too. So a round then asks a judge model, through `judge_reply`, to compare every reply, unlabelled and shuffled, and keeps the candidate only when the judge answers YES: its reply is at least as complete and correct as the best base reply. The judge also sees the shell commands each run asked for and was refused, and, when the round has it, what the person's own session ran. A run that proposed the command the work needed is then not marked down for having been refused.

In [ ]:
#| export
def route_cost(turn):
    "`(route findings, failed calls, calls)` for one turn, where refused approvals and asking again do not count and a smaller tuple is cheaper."
    a = assess_turn(turn)
    return (sum(f.kind != 'denial_retry' for f in a.findings),
            sum(x.get('ok') is False and not denial_reason(x) for x in turn.get('activity') or []), a.calls)

def _usage(turn):
    "A turn's `usage` as a dict, whether the row holds a dict or its JSON text."
    u = turn.get('usage')
    if isinstance(u, str):
        try: u = json.loads(u)
        except ValueError: u = None
    return u if isinstance(u, dict) else {}

def delivered(turn):
    "Whether the run took at least one model turn and ended with a reply and no error."
    return bool((turn.get('reply') or '').strip()) and not turn.get('error') and _usage(turn).get('turns') != 0

def better(cand, bases):
    "Whether `cand` delivered a reply and costs less than every turn in `bases`."
    return delivered(cand) and all(route_cost(cand) < route_cost(b) for b in bases)

In [ ]:
def turn(*acts, reply='done'): return {'prompt': 'sizes of the big files', 'reply': reply, 'activity': [dict(tool=t, ok=ok, args=a) for t, ok, a in acts]}
ls, ls2 = ('ls', True, {'path': ''}), ('ls', True, {'path': 'src'})
refused = ('run_shell', False, {'command': 'stat -f %z *.py'})
shell_grep = ('run_shell', True, {'command': 'grep -rn def .'})
base = turn(ls, refused, ls2)
test_eq(route_cost(base), (0, 1, 3))
test_eq(route_cost(turn(shell_grep)), (1, 0, 1))     # a shell grep is a bypass finding
unasked = dict(tool='run_shell', ok=False, args={'command': 'pytest -q'}, detail='Denied by human operator. Reason given: nothing is listening for approvals')
test_eq(route_cost(turn(ls) | {'activity': [dict(tool='ls', ok=True, args={'path': ''}), unasked]}), (0, 0, 2))   # a replay's unanswered approval is not the agent's failure
test_eq(route_cost(turn(ls) | {'activity': [unasked, unasked]}), (0, 0, 2))   # nor is asking again
assert better(turn(ls), [base, turn(ls, ls2)])
assert not better(turn(ls, ls2), [base, turn(ls, ls2)]) # ties the best base run
assert not better(turn(ls, reply=''), [base])
assert not better(turn(reply='claude/sonnet failed (ClaudeError)') | {'usage': {'turns': 0}}, [base])   # no model turn ran: the reply is the failure
assert delivered(turn(ls) | {'usage': '{"turns": 2}'}) and delivered(turn(ls) | {'usage': None})

## The learned folder

A candidate folder may hold `AGENTS.md`, up to `LEARNED_MAX` characters (2,000), and skills at `skills/<name>/SKILL.md` whose frontmatter gives the skill's `name` and a `description`. `check_learned` lists anything else as a problem, and a round does not run a candidate that has one. Hidden entries, such as `.git` and the `.kosha` index Ramabana writes into any folder it opens, are not part of the folder. `keep` replaces the learned folder's files with the candidate's and commits them in the folder's own repository, with Dhrona as the author. The first time, it commits what the folder already held before replacing it. It refuses a folder with uncommitted changes. `git revert` undoes one round.

In [ ]:
#| export
def _files(folder):
    "Every file under `folder` outside hidden folders, as relative path to text."
    folder = Path(folder)
    return {p.relative_to(folder).as_posix(): p.read_text(errors='replace') for p in sorted(folder.rglob('*'))
            if p.is_file() and not any(s.startswith('.') for s in p.relative_to(folder).parts)}

def check_learned(folder):
    "Problems that rule out a candidate learned folder; empty when it may be tried."
    out = []
    for rel, text in _files(folder).items():
        if rel == 'AGENTS.md':
            if len(text) > LEARNED_MAX: out.append(f'AGENTS.md is over {LEARNED_MAX} characters')
        elif re.fullmatch(r'skills/[a-z0-9_-]+/SKILL\.md', rel):
            name, meta = rel.split('/')[1], frontmatter(text)[0]
            if meta.get('name') != name or not meta.get('description'):
                out.append(f'{rel} needs `name: {name}` and a `description` in its frontmatter')
        else: out.append(f'{rel} is neither AGENTS.md nor skills/<name>/SKILL.md')
    return out

def _uncommitted(learned):
    "Whether the learned folder's own repository has uncommitted changes."
    learned = Path(learned)
    if not (learned/'.git').exists(): return False
    return bool(subprocess.run(['git', '-C', str(learned), 'status', '--porcelain'], capture_output=True, text=True).stdout.strip())

def keep(candidate, learned, message):
    "Replace `learned`'s files with `candidate`'s and commit them in `learned`'s own repository; returns the short commit."
    learned = Path(learned)
    learned.mkdir(parents=True, exist_ok=True)
    def git(*a): return subprocess.run(['git', '-C', str(learned), *a], check=True, capture_output=True, text=True).stdout.strip()
    def commit(msg): git('add', '-A'); git('-c', 'user.name=Dhrona', '-c', 'user.email=dhrona@localhost', 'commit', '-q', '-m', msg)
    if not (learned/'.git').exists():
        git('init', '-q')
        if _files(learned): commit('The learned folder before Dhrona kept a round')
    elif _uncommitted(learned): raise RuntimeError(f'{learned} has uncommitted changes; commit or discard them before a round keeps a candidate')
    for p in learned.iterdir():
        if not p.name.startswith('.'): shutil.rmtree(p) if p.is_dir() else p.unlink()
    shutil.copytree(candidate, learned, dirs_exist_ok=True, ignore=_HIDDEN)
    commit(message)
    return git('rev-parse', '--short', 'HEAD')

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    d = Path(d); cand, learned = d/'cand', d/'learned'
    (cand/'skills'/'sizes').mkdir(parents=True)
    (cand/'AGENTS.md').write_text('Read file sizes from `ls`; it reports them.\n')
    (cand/'skills'/'sizes'/'SKILL.md').write_text('---\nname: sizes\ndescription: Compare file sizes.\n---\nUse ls.\n')
    (cand/'.kosha').mkdir(); (cand/'.kosha'/'code.db').write_bytes(b'\x95\x00')   # the index Ramabana writes into a folder it opens
    test_eq(check_learned(cand), [])
    (cand/'notes.txt').write_text('x')
    (cand/'skills'/'bad').mkdir(); (cand/'skills'/'bad'/'SKILL.md').write_text('no frontmatter')
    test_eq(len(check_learned(cand)), 2)
    (cand/'notes.txt').unlink(); shutil.rmtree(cand/'skills'/'bad')
    (learned/'skills'/'stale').mkdir(parents=True); (learned/'skills'/'stale'/'SKILL.md').write_text('old')
    keep(cand, learned, 'Read sizes from ls')
    assert 'skills/stale/SKILL.md' in subprocess.run(['git', '-C', str(learned), 'show', '--stat', 'HEAD~1'], capture_output=True, text=True).stdout   # what was there before Dhrona is kept in history
    (learned/'AGENTS.md').write_text('a hand edit')
    test_fail(lambda: keep(cand, learned, 'again'), contains='uncommitted')
    subprocess.run(['git', '-C', str(learned), 'checkout', '-q', '--', '.'], check=True)
    test_eq(sorted(_files(learned)), ['AGENTS.md', 'skills/sizes/SKILL.md'])
    assert not (learned/'.kosha').exists()
    log = subprocess.run(['git', '-C', str(learned), 'log', '-1', '--format=%an %s'], capture_output=True, text=True).stdout.strip()
    test_eq(log, 'Dhrona Read sizes from ls')

## Evidence for the editor

The editor model reads files, not one long prompt. `trajectory_md` writes one turn as markdown: the request, each tool call with its arguments, outcome and shortened result, and the reply. `findings_md` lists each run's route cost, its findings and its failed or refused calls. `MISSION` is the editor's prompt. It says where the evidence and the candidate folder are, what the editor may change, and that changing nothing is a valid answer.

In [ ]:
#| export
def trajectory_md(turn, mx=1500):
    "One turn as markdown: the request, each tool call with its arguments, outcome and clipped result, then the reply."
    rows = [f"## Request\n\n{turn.get('prompt', '')}", '## Tool calls']
    for i, a in enumerate(turn.get('activity') or []):
        out = str(a.get('detail') or '')[:mx].replace('\n', '\n    ')
        rows.append(f"{i}. `{a.get('tool')}` {json.dumps(tool_args(a), default=str)} -> {'ok' if a.get('ok') else 'failed'}\n\n    {out}")
    rows.append(f"## Reply\n\n{turn.get('reply') or '(no reply)'}")
    return '\n\n'.join(rows)

def findings_md(turns):
    "Each run's route cost, findings, and failed or refused calls, as markdown."
    rows = []
    for n, t in enumerate(turns, 1):
        acts = t.get('activity') or []
        rows.append(f'## base-{n}: route cost {route_cost(t)} (findings, failed calls, calls)')
        rows += [f'- finding at call {f.index}: {f.message}' for f in assess_turn(t).findings]
        rows += [f"- call {i} `{a.get('tool')}` {'refused' if denial_reason(a) else 'failed'}: {str(a.get('detail') or '')[:300]}"
                 for i, a in enumerate(acts) if a.get('ok') is False]
    return '\n'.join(rows)

MISSION = r"""You improve how a coding agent, Ramabana, works by editing its learned folder, `{candidate}`.

Ramabana puts `AGENTS.md` from that folder into every briefing, after its own rules and before the project's instructions. It offers each `skills/<name>/SKILL.md` there as an on-demand skill, listed by the `name` and `description` in its frontmatter. Nothing else in the folder is read.

The evidence is in `{evidence}`. `task.md` is a request a person made. Each `base-N.md` is one run of that request under the folder as it stands. `findings.md` gives each run's route cost and what Dhrona's route check flagged. Read every evidence file and the current folder before you decide.

Look for a behaviour that repeats across the runs and costs the route something: a call that was refused or failed, a shell command where a dedicated tool does the job, or a call that repeats work already done. A defect in one run only is noise.

The runs were replays with nobody present, so every `run_shell` call was refused for want of an approver. In a real session a person approves or declines each shell command. A lesson must hold in a real session: write nothing about refused approvals, and never steer the agent away from a shell command the work needs. Redirect a shell command only when a dedicated tool does the same job. A finding is a hint, not a verdict; if no dedicated tool does what the shell command did, the finding is wrong.

`original.md`, when it is there, is the person's own session of this request, in which the person approved or declined each shell command. Its results show what a refused command in a replay would have done, and whether the person wanted it. Use it to judge whether a run's refused command was the right action. Never write a lesson that steers the agent away from a command the person approved.

Change only what the evidence shows needs changing.
- Prefer a skill for a procedure with a recognisable trigger. It costs nothing until it is used.
- Use `AGENTS.md` only for a short rule that applies to every task. Keep the whole file under {mx} characters, and cut something to pay for what you add.
- Write nothing specific to this request or this repository. The folder applies to every project.
- Never change what the agent delivers, only how it works.

If the evidence argues for no change, change nothing. That is a valid answer.

Edit the files in `{candidate}` with your tools. Then reply with one line per change naming the run and tool call that motivated it, or with the single word KEEP if you changed nothing."""

In [ ]:
print(findings_md([base]))
print(trajectory_md(base)[:400])

In [ ]:
#| export
JUDGE = r"""A person asked:

{request}

Here are {n} answers to that request from separate runs, labelled in no particular order. The runs had nobody present to approve shell commands, so each shell command a run asked to run was refused. Those commands are listed under its answer: they are what the run would have done with approval.

{answers}
{original}
Does answer {label} answer the request at least as completely and correctly as the best of the other answers? A shorter answer is fine when it is complete. An answer that stops to ask for something another answer found is less complete. A proposed command counts as part of its answer: one the request needed, stated so the person can run it, counts as done; one the request did not need, or one that would do harm, counts against it. Judge only what is written here, and do not use tools.

Reply with YES or NO on the first line, then one sentence giving the reason."""

def refused_commands(turn):
    "The shell commands `turn` asked to run and was refused for want of an approver."
    return [str(tool_args(a).get('command') or '') for a in turn.get('activity') or [] if a.get('tool') == 'run_shell' and denial_reason(a)]

def ran_commands(turn, mx=600):
    "`(command, outcome, output)` for each shell command `turn` asked for: `ran`, `failed`, or `declined: ` and the person's reason."
    return [(str(tool_args(a).get('command') or ''), f'declined: {r}' if (r := denial_reason(a)) else 'ran' if a.get('ok') else 'failed',
             '' if r else str(a.get('detail') or '')[:mx]) for a in turn.get('activity') or [] if a.get('tool') == 'run_shell']

def _commands_md(cmds): return '\n'.join(f'- `{c}`' for c in cmds)

def judge_prompt(request, cand, bases, original=None):
    "`(prompt, label)`: the judge's prompt over every reply in an order fixed by `request`, and the label of the candidate's reply."
    turns = [*bases, cand]
    order = list(range(len(turns)))
    random.Random(request).shuffle(order)
    labels = [chr(ord('A') + j) for j in range(len(order))]
    label = labels[order.index(len(turns) - 1)]
    def answer(j, i):
        asked = refused_commands(turns[i])
        return f"## Answer {labels[j]}\n\n{turns[i].get('reply') or ''}" + (f'\n\nCommands it asked to run:\n{_commands_md(asked)}' if asked else '')
    seen = ran_commands(original) if original else []
    own = ('\n## Commands in the person\'s own session of this request\n\nThe person approved or declined each one there. They show what the work needed.\n\n'
           + '\n'.join(f'- `{c}` ({how})' + (f": {' '.join(out.split())}" if out else '') for c, how, out in seen) + '\n') if seen else ''
    answers = '\n\n'.join(answer(j, i) for j, i in enumerate(order))
    return JUDGE.format(request=request, n=len(turns), answers=answers, original=own, label=label), label

def _says_yes(text):
    "Whether the first line of a judge's reply answers YES."
    return bool(re.match(r'\W*yes\b', (text or '').strip().split('\n')[0], re.I))

def judge_reply(
    request,        # what the person asked
    cand,           # the candidate run's turn
    bases,          # the base runs' turns
    folder,         # a new folder for the judge's run
    model=None,     # the model that judges
    exe=RAMABANA,
    original=None,  # the person's own turn for this request, whose commands show what the work needed
):
    "Ask a judge model whether `cand`'s reply is at least as good as the best of `bases`, and return `(kept, reply)`."
    prompt, _ = judge_prompt(request, cand, bases, original)
    folder = Path(folder)
    folder.mkdir(parents=True)
    t = run_turn(prompt, [folder], folder/'cfg', model, exe=exe)
    if not delivered(t): raise RuntimeError(f"the judge gave no answer: {t.get('error') or t.get('reply') or ''}")
    return _says_yes(t['reply']), t['reply']

A replay refuses every shell command, because nobody is there to approve it, but the command a run asked for still shows what it meant to do. `refused_commands` lists those commands, and `judge_prompt` puts them under that run's answer, so the judge can count a needed command, written out for the person to run, as done. `ran_commands` lists what the person's own session ran, what failed and what the person declined, with the reason. `judge_prompt` shows these as what the work needed, and the editor reads the same session as `original.md`.

In [ ]:
def tn(*acts, reply='done'): return {'prompt': 'tidy the drafts', 'reply': reply, 'activity': list(acts)}
look = dict(tool='ls', ok=True, args={'path': ''})
asked = dict(tool='run_shell', ok=False, args={'command': 'git add notes.md && rm draft.md'},
             detail='Denied by human operator. Reason given: nothing is listening for approvals')
broke = dict(tool='run_shell', ok=False, args={'command': 'pytest -q'}, detail='ERROR: exit 1')
ran = dict(tool='run_shell', ok=True, args={'command': 'git add notes.md && rm draft.md'}, detail='(no output)')
nope = dict(tool='run_shell', ok=False, args={'command': 'git push'}, detail='Denied by human operator. Reason given: not yet')
test_eq(refused_commands(tn(look, asked, broke)), ['git add notes.md && rm draft.md'])   # one that ran and failed was not refused
test_eq(ran_commands(tn(look, ran, broke, nope)),
        [('git add notes.md && rm draft.md', 'ran', '(no output)'), ('pytest -q', 'failed', 'ERROR: exit 1'), ('git push', 'declined: not yet', '')])
proposer = tn(look, asked, reply='Run `git add notes.md && rm draft.md`; I could not.')
avoider = tn(look, reply='Done.')
p, label = judge_prompt('tidy the drafts', proposer, [avoider], original=tn(ran))
other = 'B' if label == 'A' else 'A'
blocks = {l: p.split(f'## Answer {l}\n')[1].split('\n## ')[0] for l in 'AB'}
assert 'rm draft.md' in blocks[label] and 'rm draft.md' not in blocks[other], blocks   # under the run that asked for it
assert "person's own session" in p and '(no output)' in p
bare, _ = judge_prompt('tidy the drafts', avoider, [avoider])
assert 'Commands' not in bare and "person's own session" not in bare   # nothing refused and no session: neither section

## A round

`improve_round` takes a request and the `start` record Ramabana wrote for its session, `<cfg>/runs/<session>/start.json`, which holds `repo`, `head` and `dirty`. It raises `ValueError` before any run starts when:

- the start had uncommitted changes, which a replay from `head` would not see,
- the learned folder holds files outside the allowed layout,
- `approve` is anything but `ask` or `edits`, so a replay never runs a shell command,
- `k` is less than 1.

`work` must be a new folder, or the round raises `FileExistsError`. Every run, the evidence, the candidate and `report.json` stay in it. If a base run's model never answered, the round ends with `base_error` before the editor runs. If the editor or the candidate run records no turn or gives no answer, the round ends with that error in the report. By default replays refuse every write. `approve='edits'` lets them edit files inside their own clone, which shares no object files with the source repository. With `learned` set, a kept candidate is committed there; without it, the round only reports.

The tests below use a stand-in script for `ramabana` that answers each kind of prompt with a fixed reply. Under it, a run takes one call once `AGENTS.md` exists and three calls before.

In [ ]:
#| export
def _summary(turn):
    "What a report keeps of one turn."
    return dict(route=route_cost(turn), findings=[f.message for f in assess_turn(turn).findings],
                calls=[a.get('tool') for a in turn.get('activity') or []], reply=turn.get('reply'),
                cost=_usage(turn).get('cost'))

def improve_round(
    prompt,         # the request to replay
    start,          # `{repo, head, dirty}` from the session's `start.json`
    work,           # a new folder for this round's runs, evidence and report
    learned=None,   # the learned folder to start from and commit a kept candidate to
    model=None,     # the model the runs use
    editor=None,    # the model that edits the candidate
    k=2,            # base runs
    approve='ask',  # `edits` lets a replay edit files inside its throwaway clone
    exe=RAMABANA,
    read_outside=True, # replays read outside their clone, as a session started with `--read-outside` does
    original=None,  # the person's own turn for this request; the editor and the judge see what its commands did
):
    "Run one round of `k` base runs, one edit to the learned folder, one run under the edit and the decision, and return the report."
    if k < 1: raise ValueError('a round needs at least one base run')
    if approve not in ('ask', 'edits'): raise ValueError(f"approve is 'ask' or 'edits', not {approve!r}: a replay never runs shell commands")
    if start.get('dirty'): raise ValueError(f"{start['repo']} had uncommitted changes when the session began; a replay from {start['head']} would not see them")
    work = Path(work)
    if work.exists(): raise FileExistsError(f'{work} exists; a round writes into a new folder')
    current, cand, ev = work/'current', work/'candidate', work/'evidence'
    if learned is not None and Path(learned).exists(): shutil.copytree(learned, current, ignore=_HIDDEN)
    else: current.mkdir(parents=True)
    if (bad := check_learned(current)): raise ValueError(f'{learned} holds files a learned folder may not: {bad}')
    def run(name, folder): return run_turn(prompt, [replay_clone(start, work/name/'repo')], work/name/'cfg', model, folder, approve, exe=exe,
                                           read_outside=read_outside)
    bases = [run(f'base-{n}', current) for n in range(1, k+1)]
    report = dict(prompt=prompt, start=start, bases=[_summary(b) for b in bases], adopted=False)
    if (dead := [n for n, b in enumerate(bases, 1) if not delivered(b)]):
        first = bases[dead[0] - 1]
        report['base_error'] = f"base runs {dead} gave no answer: {(first.get('error') or first.get('reply') or '')[:300]}"
        (work/'report.json').write_text(json.dumps(report, indent=1, default=str))
        return report
    ev.mkdir()
    (ev/'task.md').write_text(prompt)
    for n, b in enumerate(bases, 1): (ev/f'base-{n}.md').write_text(trajectory_md(b))
    (ev/'findings.md').write_text(findings_md(bases))
    if original: (ev/'original.md').write_text("# The person's own session of this request\n\nA person approved or declined each shell command here.\n\n" + trajectory_md(original))
    shutil.copytree(current, cand, ignore=_HIDDEN)
    try: edit = run_turn(MISSION.format(candidate=cand, evidence=ev, mx=LEARNED_MAX), [cand, ev], work/'editor'/'cfg', editor,
                         approve='edits', exe=exe)
    except RuntimeError as e: edit = dict(error=str(e))
    editor_error = edit.get('error') or (None if delivered(edit) else f"the editor gave no answer: {edit.get('reply') or ''}")
    report |= dict(notes=edit.get('reply'), editor_error=editor_error, editor_cost=_usage(edit).get('cost'),
                   changed=_files(cand) != _files(current), problems=check_learned(cand))
    if report['changed'] and not report['problems'] and not report['editor_error']:
        try: c = run('candidate-run', cand)
        except RuntimeError as e: c = None; report['candidate_error'] = str(e)
        if c is not None:
            report['candidate'] = _summary(c)
            if better(c, bases):   # a cheaper route counts only if the answer is as good
                try: report['adopted'], report['judge'] = judge_reply(prompt, c, bases, work/'judge', editor, exe, original)
                except RuntimeError as e: report['judge_error'] = str(e)
            if report['adopted'] and learned is not None:
                routes = ', '.join(str(b['route']) for b in report['bases'])
                msg = f"Lesson from a replay: {' '.join(prompt.split())[:60]}\n\n{report['notes']}\n\nRoute cost (findings, failed calls, calls): base {routes}; candidate {report['candidate']['route']}."
                report['commit'] = keep(cand, learned, msg)
    (work/'report.json').write_text(json.dumps(report, indent=1, default=str))
    return report

In [ ]:
FAKE = r'''#!/usr/bin/env python3
"A stand-in for `ramabana`: the editor writes AGENTS.md; a run takes one call once AGENTS.md exists, three before."
import json, sys
from pathlib import Path
a = sys.argv[1:]
roots, cfg, prompt = a[a.index('--root') + 1].split(','), Path(a[a.index('--cfg') + 1]), a[-1]
ls = dict(tool='ls', ok=True, args={'path': ''})
if prompt.startswith('You improve'):
    Path(roots[0], 'AGENTS.md').write_text('Read file sizes from ls.\n')
    acts, reply = [dict(tool='create_file', ok=True, args={})], 'base-1 call 1: wrote AGENTS.md'
elif prompt.startswith('A person asked'): acts, reply = [], 'YES\nIt names the same file and size.'
elif (cfg/'learned'/'AGENTS.md').exists(): acts, reply = [ls], 'big.py 50'
else: acts, reply = [ls, dict(tool='run_shell', ok=False, args={'command': 'stat big.py'}), ls], 'big.py 50'
(cfg/'agent-history.jsonl').write_text(json.dumps(dict(prompt=prompt, activity=acts, reply=reply, usage={'cost': 0})) + '\n')
'''
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo = d/'fake-ramabana', d/'repo'
    exe.write_text(FAKE); exe.chmod(0o755); repo.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    start = dict(repo=str(repo), head=git('rev-parse', 'HEAD'), dirty=[])
    r = improve_round('sizes of the big files', start, d/'work', learned=d/'learned', exe=str(exe))
    test_eq((r['bases'][0]['route'], r['candidate']['route'], r['adopted']), ((0, 1, 3), (0, 0, 1), True))
    assert r['judge'].startswith('YES')
    test_eq(subprocess.run(['git', '-C', str(d/'work'/'base-1'/'repo'), 'remote'], capture_output=True, text=True).stdout, '')   # nothing can push back
    test_eq(sorted(_files(d/'learned')), ['AGENTS.md'])
    test_eq(sorted(_files(d/'work'/'candidate')), ['AGENTS.md'])   # the runs sit beside the candidate, never in it
    assert (d/'work'/'report.json').exists()
    assert not (d/'work'/'evidence'/'original.md').exists()
    real = dict(prompt='sizes of the big files', reply='big.py 50', activity=[dict(tool='run_shell', ok=True, args={'command': 'stat -f %z big.py'}, detail='50')])
    r = improve_round('sizes of the big files', start, d/'w7', exe=str(exe), original=real)
    assert 'stat -f %z big.py' in (d/'w7'/'evidence'/'original.md').read_text()   # the editor reads what the person's session ran
    test_fail(lambda: improve_round('x', start, d/'work', exe=str(exe)), contains='exists')
    test_fail(lambda: improve_round('x', start | dict(dirty=['big.py']), d/'w2', exe=str(exe)), contains='uncommitted')
    test_fail(lambda: improve_round('x', start, d/'w3', approve='auto', exe=str(exe)), contains='approve')
    doubting = d/'doubting-ramabana'
    doubting.write_text(FAKE.replace("'YES\\nIt names", "'NO\\nIt asks instead of naming")); doubting.chmod(0o755)
    r = improve_round('sizes of the big files', start, d/'w5', learned=d/'learned-5', exe=str(doubting))
    test_eq((r['candidate']['route'], r['adopted']), ((0, 0, 1), False))   # a cheaper route that answers worse is not kept
    assert r['judge'].startswith('NO') and not (d/'learned-5').exists()
    mute = d/'mute-ramabana'
    mute.write_text(FAKE.replace("'YES\\nIt names the same file and size.'", "''")); mute.chmod(0o755)
    r = improve_round('sizes of the big files', start, d/'w6', exe=str(mute))
    assert 'no answer' in r['judge_error'] and not r['adopted']
    assert all(map(_says_yes, ['YES', '**YES**', 'yes: it names the file'])) and not any(map(_says_yes, ['YESTERDAY', 'NO', 'Answer C: YES']))
    dead = d/'dead-ramabana'
    dead.write_text(FAKE.replace("usage={'cost': 0}", "usage={'turns': 0}")); dead.chmod(0o755)
    r = improve_round('sizes of the big files', start, d/'w4', exe=str(dead))
    assert 'no answer' in r['base_error'] and 'big.py 50' in r['base_error'] and not (d/'w4'/'editor').exists()   # no editor runs on empty evidence

## Training on past sessions

Older sessions have no `start.json`, so they record no starting commit, and most record no folder either. `session_start` works both out. It places a turn in the repository that most of its absolute tool paths fall in, or else the first git repository among `roots`. Its `head` is that repository's last commit before the turn's `at`, which may be epoch seconds or an ISO date; an unreadable `at` leaves the turn unplaced. `dirty` is None because nobody knows what was uncommitted at that time.

`trainable` picks the turns worth replaying. It takes the first turn of each session when the request is at least 40 characters long, carried no attachments, ended without an error, made between 1 and `max_calls` calls, and cost something, meaning at least one finding or failed call. An attached file belonged only to that session, so a replay could not see it. It also drops a turn that made a commit, because `at` marks the end of the turn and the commit found for its start would already contain that commit. A request asked in several sessions is kept once, from the cheapest, and the result is ordered by number of calls, fewest first. `trainable_sessions` runs `trainable` over Ramabana's own history and session index, takes each session's start from its `start.json` entry for the repository the work used, and drops a session whose entry had uncommitted changes.

In [ ]:
#| export
def request_text(prompt):
    "The person's request inside `<user-request>` tags, else the whole prompt."
    found = re.findall(r'<user-request>\n?(.*?)\n?</user-request>', str(prompt or ''), re.S)
    return (found[-1] if found else str(prompt or '')).strip()

def _toplevel(path):
    "The git repository holding the absolute `path`, after `~` expansion, or None."
    p = Path(path).expanduser()
    if not p.is_absolute(): return None
    return next((str(d) for d in [p, *p.parents] if (d/'.git').exists()), None)

def _when(at):
    "A history row's `at`, an epoch in seconds or an ISO date, as an ISO date git understands; None when unreadable."
    try: return datetime.fromtimestamp(float(at), timezone.utc).isoformat()
    except (TypeError, ValueError, OverflowError, OSError): pass
    try: return datetime.fromisoformat(str(at)).isoformat()
    except ValueError: return None

def session_start(turn, roots=()):
    "`{repo, head, dirty, inferred}` for replaying `turn`'s session, or None when nothing places it in a repository."
    hits = Counter()
    for a in turn.get('activity') or []:
        for k in ('path', 'cwd', 'file', 'root'):
            if (r := _toplevel(str(tool_args(a).get(k) or ''))): hits[r] += 1
    repo = hits.most_common(1)[0][0] if hits else next((r for r in map(_toplevel, roots) if r), None)
    if repo is None or (when := _when(turn.get('at'))) is None: return None
    head = subprocess.run(['git', '-C', repo, 'rev-list', '-1', f'--before={when}', 'HEAD'], capture_output=True, text=True).stdout.strip()
    return dict(repo=repo, head=head, dirty=None, inferred=True) if head else None

def trainable(turns, roots_of=None, max_calls=40):
    "The first turn of each session that can seed a round, shortest first: `{session, prompt, model, start, route, turn}`."
    first = {}
    for t in turns:
        if t.get('session'): first.setdefault(t['session'], t)
    found = []
    for sid, t in first.items():
        req, cost = request_text(t.get('prompt')), route_cost(t)
        if len(req) < 40 or '<attachments>' in str(t.get('prompt')) or t.get('error'): continue   # an attachment is a file only that session had
        if not 0 < cost[2] <= max_calls or cost[:2] == (0, 0): continue
        if any(a.get('tool') == 'git_commit' for a in t.get('activity') or []): continue    # `at` is the turn's end, so its start would hold this commit
        if (start := session_start(t, (roots_of or {}).get(sid) or ())) is None: continue
        found.append(dict(session=sid, prompt=req, model=t.get('model'), start=start, route=cost, turn=t))
    out, seen = [], set()
    for s in sorted(found, key=lambda s: s['route'][2]):
        if s['prompt'] not in seen: seen.add(s['prompt']); out.append(s)
    return out

def trainable_sessions(cfg='~/.config/ramabana', max_calls=40):
    "`trainable` over Ramabana's history and session index in `cfg`, with each session's own `start.json` where it has a clean one."
    cfg = Path(cfg).expanduser()
    idx = cfg/'agent-sessions.json'
    roots = {s: v.get('roots') for s, v in json.loads(idx.read_text()).get('sessions', {}).items()} if idx.exists() else {}
    out = []
    for s in trainable(read_history(cfg/'agent-history.jsonl'), roots, max_calls):
        p = cfg/'runs'/s['session']/'start.json'
        rec = next((r for r in (json.loads(p.read_text()) if p.exists() else []) if r.get('repo') == s['start']['repo']), None)
        if rec is not None:
            if rec.get('dirty'): continue
            s['start'] = rec | dict(inferred=False)
        out.append(s)
    return out

In [ ]:
import os
from datetime import datetime, timezone
with TemporaryDirectory() as d:
    repo = Path(d)/'repo'; repo.mkdir()
    def git(*a, when=None):
        env = os.environ | ({'GIT_AUTHOR_DATE': when, 'GIT_COMMITTER_DATE': when} if when else {})
        return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True, env=env).stdout.strip()
    git('init', '-q')
    for n, when in [(1, '2026-01-01T00:00:00Z'), (2, '2026-03-01T00:00:00Z')]:
        (repo/'a.py').write_text(f'x = {n}\n'); git('add', '.')
        git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', f'c{n}', when=when)
    january = git('rev-list', '--max-parents=0', 'HEAD')
    t = {'session': 's1', 'at': datetime(2026, 2, 1, tzinfo=timezone.utc).timestamp(), 'model': 'claude-opus-5-5',
         'prompt': '<user-request>\nwhere is x defined, and which files read it? list them\n</user-request>',
         'activity': [dict(tool='grep', ok=True, args={'pattern': 'x', 'path': str(repo/'a.py')}),
                      dict(tool='run_shell', ok=False, args={'command': 'cat a.py'})]}
    test_eq((session_start(t)['repo'], session_start(t)['head']), (str(repo), january))   # the last commit before the turn
    test_eq(session_start(t | {'at': '2026-02-01T00:00:00+00:00'})['head'], january)        # an ISO `at` reads the same
    test_eq((session_start(t | {'activity': []}), session_start(t | {'at': None}), _toplevel('relative/dir')), (None, None, None))
    test_eq(session_start(t | {'activity': []}, roots=[str(repo)])['repo'], str(repo))
    attached = t | {'session': 's3', 'prompt': '<user-request>\nwhat does this screenshot show about the layout?\n</user-request>\n<attachments>\nimage /tmp/paste.png\n</attachments>'}
    cheaper = t | {'session': 's5', 'activity': [dict(tool='grep', ok=False, args={'pattern': 'x', 'path': str(repo/'a.py')})]}
    committed = t | {'session': 's6', 'prompt': 'where is x defined, and who imports it? list each file please',
                     'activity': t['activity'] + [dict(tool='git_commit', ok=True, args={'message': 'm'})]}
    got = trainable([t, t | {'prompt': 'a later turn in the same session'}, t | {'session': 's2', 'prompt': 'ok'},
                     t | {'session': 's4'}, attached, cheaper, committed])
    test_eq([(s['session'], s['prompt'], s['model']) for s in got], [('s5', 'where is x defined, and which files read it? list them', 'claude-opus-5-5')])
    test_eq(got[0]['turn']['session'], 's5')   # the session's own first turn, for the editor and the judge
    # s4 and s5 ask what s1 asked and s5 costs least; s3 names a file only its session had; s6's start would hold its own commit
    cfg = Path(d)/'cfg'; (cfg/'runs'/'s1').mkdir(parents=True)
    (cfg/'agent-history.jsonl').write_text(json.dumps(t) + '\n')
    rec = [dict(repo='/elsewhere', head='abc', dirty=[]), dict(folder=str(repo), repo=str(repo), head=january, dirty=[])]
    (cfg/'runs'/'s1'/'start.json').write_text(json.dumps(rec))
    test_eq([(s['start']['head'], s['start']['inferred']) for s in trainable_sessions(cfg)], [(january, False)])   # the entry for the repository the work used
    rec[1]['dirty'] = ['a.py']; (cfg/'runs'/'s1'/'start.json').write_text(json.dumps(rec))
    test_eq(trainable_sessions(cfg), [])

## Batches

`improve_batch` runs `improve_round` on up to `n` sessions in turn, in the order `trainable_sessions` gives. Each round starts from the learned folder the previous round left, so a later round sees an earlier round's lesson. `models` keeps the sessions recorded with those models, and `skip` leaves sessions out by id. A round that fails, for example because its model cannot start or its repository is gone, becomes a row with `error`, and the batch goes on. `summary.json` in `work` holds every row so far, and `[]` when no session matched. An `approve` other than `ask` or `edits`, or a learned folder with uncommitted changes, raises before any round runs.

Each row has a `cost`: the dollars recorded by every run in the round's folder (`round_cost`), meaning the base runs, the editor, the candidate and the judge, whether or not the round finished. A run killed at its timeout, or one whose model reports no cost, adds nothing. With `budget` set, no new round starts once the rounds so far have cost that much, so a batch can overshoot its budget by the cost of its last round. A batch with a learned folder holds `learned_lock` on it for its whole run, and a second batch on the same folder, from the command line or a scheduled beat, raises `RuntimeError` before it creates anything. The lock is a file beside the folder, and the operating system releases it when the batch's process ends, even in a crash. `dhrona-improve` runs a batch from the command line. It requires a learned folder, because the rounds of a batch build on each other through it.

In [ ]:
#| export
def round_cost(work):
    "Dollars the runs in round folder `work` recorded, whether or not the round finished; a run that recorded no turn adds nothing."
    return sum(c for h in Path(work).glob('*/cfg/agent-history.jsonl') if (ts := read_history(h)) and (c := _usage(ts[-1]).get('cost')))

@contextmanager
def learned_lock(learned):
    "Hold `learned`'s lock for the block, or raise `RuntimeError` at once when another batch holds it."
    learned = Path(learned)
    learned.parent.mkdir(parents=True, exist_ok=True)
    with open(learned.parent/f'.{learned.name}.lock', 'w') as f:   # beside the folder: inside it, `git status` would count it as a change
        try:
            if os.name == 'nt': import msvcrt; msvcrt.locking(f.fileno(), msvcrt.LK_NBLCK, 1)
            else: import fcntl; fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError: raise RuntimeError(f'another batch is using {learned}') from None
        yield

def improve_batch(
    work,              # a new folder for one round folder per session and `summary.json`
    learned=None,      # the learned folder each round starts from and commits a kept candidate to
    sessions=None,     # `trainable_sessions()` when None
    n=10,              # rounds to run
    model=None,        # replay every session with this model; None uses each session's own
    editor=None,       # the model that edits each candidate
    models=None,       # keep only sessions recorded with one of these models
    skip=(),           # session ids to leave out
    k=2,               # base runs per round
    approve='edits',   # `ask` refuses every write in a replay
    exe=RAMABANA,      # the Ramabana command a round runs
    read_outside=True, # replays read outside their clone
    budget=None,       # dollars; no round starts once the rounds so far have cost this much
):
    "Run `improve_round` on up to `n` sessions in turn; returns one row per round, also written to `work/summary.json`."
    work = Path(work)
    if work.exists(): raise FileExistsError(f'{work} exists; a batch writes into a new folder')
    if approve not in ('ask', 'edits'): raise ValueError(f"approve is 'ask' or 'edits', not {approve!r}: a replay never runs shell commands")
    with learned_lock(learned) if learned is not None else nullcontext():
        if learned is not None and _uncommitted(learned): raise RuntimeError(f'{learned} has uncommitted changes; commit or discard them before a batch')
        picked = [s for s in (trainable_sessions() if sessions is None else sessions)
                  if s['session'] not in skip and (models is None or s['model'] in models)][:n]
        work.mkdir(parents=True)
        rows, spent = [], 0
        (work/'summary.json').write_text('[]')
        for s in picked:
            if budget is not None and spent >= budget: break
            row = dict(session=s['session'], prompt=s['prompt'], model=model or s['model'])
            try:
                r = improve_round(s['prompt'], s['start'], work/s['session'], learned, row['model'], editor, k, approve, exe, read_outside,
                                  original=s.get('turn'))
                row |= dict(adopted=r['adopted'], commit=r.get('commit'), changed=r.get('changed'), problems=r.get('problems'),
                            bases=[b['route'] for b in r['bases']], candidate=(r.get('candidate') or {}).get('route'), judge=r.get('judge'),
                            error=r.get('base_error') or r.get('editor_error') or r.get('candidate_error') or r.get('judge_error') or None)
            except (RuntimeError, ValueError, OSError, subprocess.CalledProcessError) as e: row['error'] = f'{type(e).__name__}: {e}'
            row['cost'] = round_cost(work/s['session'])
            spent += row['cost']
            rows.append(row)
            (work/'summary.json').write_text(json.dumps(rows, indent=1, default=str))
    return rows

@call_parse
def improve_cli(
    work:str,                       # a new folder for this batch
    learned:str,                    # the learned folder to start from and commit kept lessons to
    n:int=10,                       # rounds to run
    k:int=2,                        # base runs per round
    model:str=None,                 # replay every session with this model instead of its own
    editor:str=None,                # the model that edits each candidate
    models:str='',                  # comma-separated: replay only sessions recorded with these models
    skip:str='',                    # comma-separated session ids to leave out
    approve:str='edits',            # `ask` refuses every write in a replay
    cfg:str='~/.config/ramabana',   # the Ramabana config folder whose history is read
    max_calls:int=40,               # replay only sessions whose first turn made at most this many calls
    exe:str=RAMABANA,               # the Ramabana command a round runs
    read_outside:bool=True,         # --no-read-outside keeps replays' reads inside their clone
    budget:float=None,              # dollars: no round starts once the batch has cost this much
):
    "Run up to `n` improvement rounds on past Ramabana sessions and print one line per round, then what they cost."
    rows = improve_batch(work, learned, trainable_sessions(cfg, max_calls), n, model, editor, [m for m in models.split(',') if m] or None,
                         [s for s in skip.split(',') if s], k, approve, exe, read_outside, budget)
    if not rows: print('no session matched')
    for r in rows:
        if r.get('error'): out = r['error']
        elif r.get('adopted'): out = f"kept {r['commit']}"
        elif r.get('problems'): out = f"not kept: {r['problems'][0]}"
        elif not r.get('changed'): out = 'not kept: the editor changed nothing'
        elif r.get('judge'): out = 'not kept: the judge preferred a base answer'
        else: out = 'not kept: the candidate was no cheaper'
        print(f"{r['session']}  {r['model']}  {out}  {r['prompt'][:60]}")
    if rows: print(f"spent ${sum(r['cost'] for r in rows):.2f}")

## Every night

`nightly` is the function a scheduled beat calls. `ramabana-tick` finds it through the `ramabana.ticks` entry point named `dhrona`, and calls it when the pobblebonk schedule named `dhrona` fires, passing the schedule's payload as `fire`. The payload must hold a positive `budget` in dollars, or `nightly` raises `ValueError` before any round runs. The other keys are optional:

| key | meaning | default |
|---|---|---|
| `n` | rounds | 3 |
| `k` | base runs per round | 2 |
| `model` | the model the replays use | each session's own |
| `editor` | the model that edits and judges | Ramabana's default |
| `models` | a list of model ids whose sessions are kept | all |
| `max_calls` | the most calls a replayed turn may have made | 40 |
| `cfg` | the Ramabana config folder | `~/.config/ramabana` |
| `learned` | the learned folder | `<cfg>/learned`, the folder Ramabana loads |
| `work` | where batch folders go | `NIGHTLY`, `~/.cache/dhrona/nightly` |
| `exe` | the Ramabana command | see below |

Each night writes a new batch folder under `work`, named by the time it started. It skips every session that a batch under `work` has already started a round on (`tried_sessions`), including rounds that failed, and when no session is left it creates nothing. `nightly` returns the note the beat leaves for the next session: how many rounds ran, the lessons kept, and the dollars spent against the budget.

If a batch is already running on the learned folder, the night raises `RuntimeError`. pobblebonk retries a fire that raised, up to the schedule's `retries`, so add the schedule with `retries=1` and a refused or failed night waits for the next one:

```python
from ramabana.monitor import pob
pob().add('dhrona', cron='0 3 * * *', retries=1, budget=20, n=3)
```

Without `exe`, a night runs the `ramabana` installed beside the Python that runs the beat, or else the one found on `PATH`. This matters because a beat started by cron or launchd has a short `PATH`.

In [ ]:
#| export
NIGHTLY = '~/.cache/dhrona/nightly'   #: where scheduled batches write, one folder a night

def tried_sessions(root):
    "Session ids that the batches under `root` have already started a round on."
    return {p.name for p in Path(root).expanduser().glob('*/*') if p.is_dir()}

def _beside_python():
    "The `ramabana` installed beside this interpreter, else the name for `PATH` to find."
    return shutil.which(RAMABANA, path=str(Path(sys.executable).parent)) or RAMABANA

def nightly(fire):
    "One scheduled batch over sessions no earlier night tried, within `fire.budget` dollars; returns the note the beat leaves."
    if not (fire.get('budget') or 0) > 0: raise ValueError('the dhrona schedule needs a positive `budget` in dollars')
    cfg, root = fire.get('cfg', '~/.config/ramabana'), Path(fire.get('work', NIGHTLY)).expanduser()
    tried = tried_sessions(root)
    left = [s for s in trainable_sessions(cfg, fire.get('max_calls', 40)) if s['session'] not in tried]
    if not left: return 'no session left that an earlier night has not tried'
    work = root/datetime.now().strftime('%Y-%m-%d-%H%M%S')
    rows = improve_batch(work, Path(fire.get('learned', Path(cfg)/'learned')).expanduser(), left, n=fire.get('n', 3), k=fire.get('k', 2),
                         model=fire.get('model'), editor=fire.get('editor'), models=fire.get('models'), exe=fire.get('exe') or _beside_python(),
                         budget=fire.budget)
    kept, failed = [r for r in rows if r.get('adopted')], [r for r in rows if r.get('error')]
    head = f"{len(rows)} rounds, {len(kept)} kept, {len(failed)} failed, ${sum(r['cost'] for r in rows):.2f} of ${fire.budget}; see {work}"
    return '\n'.join([head, *(f"kept {r['commit']}: {r['prompt'][:60]}" for r in kept)])

In [ ]:
import time
from fastcore.basics import AttrDict

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo, cfg = d/'paid-ramabana', d/'repo', d/'cfg'
    exe.write_text(FAKE.replace("usage={'cost': 0}", "usage={'cost': 1}")); exe.chmod(0o755); repo.mkdir(); cfg.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    ls = dict(tool='ls', ok=True, args={'path': str(repo)})
    past = dict(session='s1', model='m1', at=time.time() + 60, reply='big.py 50',
                prompt='<user-request>\nWhat are the sizes of the big files in this repository?\n</user-request>',
                activity=[ls, dict(tool='run_shell', ok=False, args={'command': 'stat big.py'}), ls])
    (cfg/'agent-history.jsonl').write_text(json.dumps(past) + '\n')
    fire = AttrDict(budget=5, cfg=str(cfg), work=str(d/'nights'), exe=str(exe))
    note = nightly(fire)
    print(note)
    assert note.startswith('1 rounds, 1 kept, 0 failed, $5.00 of $5') and (cfg/'learned'/'AGENTS.md').exists()
    test_eq(tried_sessions(d/'nights'), {'s1'})
    test_eq(nightly(fire), 'no session left that an earlier night has not tried')
    test_eq(len(list((d/'nights').iterdir())), 1)   # the second night created no folder
    test_fail(lambda: nightly(AttrDict(cfg=str(cfg))), contains='budget')

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo = d/'fake-ramabana', d/'repo'
    exe.write_text(FAKE); exe.chmod(0o755); repo.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    start = dict(repo=str(repo), head=git('rev-parse', 'HEAD'), dirty=None)
    def session(sid, model='m1', start=start): return dict(session=sid, prompt=f'sizes of the big files ({sid})', model=model, start=start, route=(0, 1, 3))
    real = dict(prompt='sizes of the big files', reply='big.py 50', activity=[dict(tool='run_shell', ok=True, args={'command': 'stat big.py'}, detail='50')])
    sessions = [session('s1') | dict(turn=real), session('s2'), session('s3', model='local'), session('s4', start=start | dict(repo=str(d/'gone')))]
    rows = improve_batch(d/'batch', learned=d/'learned', sessions=sessions, models=['m1'], exe=str(exe))
    test_eq([(r['session'], r.get('adopted')) for r in rows], [('s1', True), ('s2', False), ('s4', None)])   # s2 starts from s1's lesson; s3's model is not offered
    assert rows[-1]['error'].startswith('RuntimeError: could not clone') and 'does not exist' in rows[-1]['error']
    assert (d/'batch'/'s1'/'evidence'/'original.md').exists() and not (d/'batch'/'s2'/'evidence'/'original.md').exists()   # each round gets its own session
    test_eq(json.loads((d/'batch'/'summary.json').read_text()), json.loads(json.dumps(rows, default=str)))
    test_eq(improve_batch(d/'none', learned=d/'learned', sessions=sessions, models=['absent'], exe=str(exe)), [])
    test_eq(json.loads((d/'none'/'summary.json').read_text()), [])
    test_fail(lambda: improve_batch(d/'bad', learned=d/'learned', sessions=sessions, approve='auto', exe=str(exe)), contains='approve')
    paid = d/'paid-ramabana'
    paid.write_text(FAKE.replace("usage={'cost': 0}", "usage={'cost': 1}")); paid.chmod(0o755)
    rows = improve_batch(d/'capped', learned=d/'learned-capped', sessions=sessions, budget=5, exe=str(paid))
    test_eq([(r['session'], r['cost']) for r in rows], [('s1', 5)])   # two bases, the editor, the candidate and the judge; then the cap stops the batch
    paid.write_text(FAKE.replace("usage={'cost': 0}", "usage={'cost': 1}").replace("'YES\\nIt names the same file and size.'", "''"))
    rows = improve_batch(d/'mute-judge', learned=d/'learned-mute', sessions=sessions[:1], budget=5, exe=str(paid))
    test_eq((rows[0]['cost'], 'judge gave no answer' in rows[0]['error']), (5, True))   # a judge that gave no answer was still paid for
    with learned_lock(d/'learned'):
        test_fail(lambda: improve_batch(d/'locked', learned=d/'learned', sessions=sessions, exe=str(exe)), contains='another batch')
    (d/'learned'/'AGENTS.md').write_text('a hand edit')
    test_fail(lambda: improve_batch(d/'dirty', learned=d/'learned', sessions=sessions, exe=str(exe)), contains='uncommitted')
    assert not any((d/f).exists() for f in ('bad', 'dirty', 'locked'))   # refused before any round runs

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()